# Img2GPS — DINOv2 retrieval (Colab + local)

**Submit to the leaderboard:** `Img2GPS/model.py`, `Img2GPS/preprocess.py`, `Img2GPS/model.pt`.

**Model:** frozen **DINOv2 ViT-S/14** + **soft retrieval** — one embedding and raw `[lat, lon]` per training row in `metadata.csv`, plus a **temperature** scalar fitted by **mean Haversine** on a location holdout (`train.py`). Gallery = **all** your photos when you run the train cell with `use_all_data=True`.

**Sections:** (0) Bootstrap repo & deps · (1) Data sanity · (2) Train & download `model.pt` · (3) Run `eval_project_a.py` · (4) Optional scatter on the reference set.

**Lighting:** rely on a strong pretrained encoder; add photos at different times of day if exposure varies a lot.


## 0. Bootstrap (Colab + local)

Colab: sparse-clone the repo's code (skipping the ~4.7 GB `data/images_converted/` checkout) into `/content/cis_5190_project`, install `requirements.txt` plus `huggingface_hub`. Local: run from repo root or any subfolder — the next cell walks up to find `Img2GPS/`. Afterward `PROJECT_DIR` points at `Img2GPS/` and `train`, `preprocess`, `model` import from there.

Training images and `metadata.csv` come from the [`joduman/spruce`](https://huggingface.co/datasets/joduman/spruce) dataset on Hugging Face via `snapshot_download` (cached after the first call). `HF_TRAIN_CSV` later in the notebook points at the HF copy; `preprocess.py`'s candidate-path resolver finds the matching images at `<DATA_DIR>/images/`.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/SheilaBkny/cis_5190_project.git'
REPO_BRANCH = 'main'
COLAB_REPO_DIR = '/content/cis_5190_project'
HF_DATASET_REPO = 'joduman/spruce'

IN_COLAB = 'google.colab' in sys.modules

def _git(*args):
    subprocess.run(['git', '-C', COLAB_REPO_DIR, *args], check=True)

if IN_COLAB:
    if not os.path.exists(os.path.join(COLAB_REPO_DIR, 'Img2GPS')):
        # Sparse clone: the heavy data/images_converted/ tree (~4.7 GB) is
        # skipped here -- training images come from the Hugging Face dataset
        # (joduman/spruce) below.
        subprocess.run([
            'git', 'clone',
            '--filter=blob:none', '--sparse',
            '--branch', REPO_BRANCH,
            REPO_URL, COLAB_REPO_DIR,
        ], check=True)
        _git(
            'sparse-checkout', 'set',
            'Img2GPS', 'requirements.txt', 'SUBMISSION.md', 'CIS5190_Img2GPS_Report.tex',
        )
    else:
        _git('fetch', 'origin', REPO_BRANCH)
        _git('checkout', REPO_BRANCH)
        _git('reset', '--hard', f'origin/{REPO_BRANCH}')
    os.chdir(COLAB_REPO_DIR)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)
    head = subprocess.check_output(['git', '-C', COLAB_REPO_DIR, 'log', '-1', '--oneline']).decode().strip()
    print(f'on commit: {head}')

import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd()
while REPO_ROOT.name and not (REPO_ROOT / 'Img2GPS').exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'Img2GPS').exists():
    raise RuntimeError(f'could not locate Img2GPS/ from cwd={Path.cwd()}; bootstrap failed')
PROJECT_DIR = REPO_ROOT / 'Img2GPS'
sys.path.insert(0, str(PROJECT_DIR))
os.chdir(REPO_ROOT)

# Pull metadata.csv + images/ from joduman/spruce on Hugging Face. The
# snapshot is cached so subsequent runs are near-instant.
try:
    from huggingface_hub import snapshot_download
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)
    from huggingface_hub import snapshot_download

DATA_DIR = Path(snapshot_download(repo_id=HF_DATASET_REPO, repo_type='dataset'))
HF_TRAIN_CSV = DATA_DIR / 'metadata.csv'
if not HF_TRAIN_CSV.exists():
    raise RuntimeError(f'metadata.csv not found inside HF dataset at {DATA_DIR}')

if torch.cuda.is_available():
    device = torch.device('cuda')
elif getattr(torch.backends, 'mps', None) is not None and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print('torch     :', torch.__version__)
print('device    :', device)
print('repo root :', REPO_ROOT)
print('project   :', PROJECT_DIR)
print('hf dataset:', DATA_DIR)
print('hf csv    :', HF_TRAIN_CSV)

## 1. Data sanity check

Loads the Hugging Face copy of `metadata.csv` and the local `reference/` CSV with `preprocess.load_raw`. The reference set is small and stays in the repo; the training set is whatever lives in [`joduman/spruce`](https://huggingface.co/datasets/joduman/spruce). No separate training dataset class — `train.py` reads the same tensors.


In [ ]:
from preprocess import load_raw  # noqa: E402

TRAIN_CSV = str(HF_TRAIN_CSV)  # metadata.csv from joduman/spruce on Hugging Face
REF_CSV = str(PROJECT_DIR / 'reference' / 'metadata.csv')

X_train_raw, y_train = load_raw(TRAIN_CSV)
X_ref_raw, y_ref = load_raw(REF_CSV)

print('train :', tuple(X_train_raw.shape), tuple(y_train.shape))
print('reference :', tuple(X_ref_raw.shape), tuple(y_ref.shape))
print()
print(pd.read_csv(TRAIN_CSV).head(3).to_string(index=False))


### Optional: preview a few training images (raw lat/lon in title)


In [ ]:
import matplotlib.pyplot as plt

n_show = min(4, X_train_raw.shape[0])
fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 3))
if n_show == 1:
    axes = [axes]
for i in range(n_show):
    img = X_train_raw[i].numpy().transpose(1, 2, 0)
    la, lo = float(y_train[i, 0]), float(y_train[i, 1])
    axes[i].imshow(img)
    axes[i].set_title(f'{la:.5f}\n{lo:.5f}', fontsize=8)
    axes[i].axis('off')
plt.tight_layout()
plt.show()


## 2. Build `model.pt` and download (DINOv2 retrieval)

Runs `train.train(...)` in this kernel (same as `python Img2GPS/train.py --csv <HF_TRAIN_CSV>`). The CSV comes from `joduman/spruce`; image paths resolve via `preprocess.py`'s candidates (`<DATA_DIR>/images/<file>` is the one that hits). First run may download ~85 MB DINOv2 weights.

- **`BOOTSTRAP_ROUNDS`:** e.g. `25` for mean±std OOB Haversine; set `0` before the final submission run.
- **`temp_steps`:** Adam steps for Haversine minimization over temperature `T` (default 800).
- **Gallery:** `use_all_data=True` keeps every CSV row in the gallery; `T` still uses a location holdout.


In [ ]:
import hashlib
import sys
from pathlib import Path

import train as train_mod  # noqa: E402  — lives in Img2GPS/ (on sys.path from bootstrap)

TRAIN_CSV = str(HF_TRAIN_CSV)  # metadata.csv from joduman/spruce on Hugging Face
OUT_PT = str(PROJECT_DIR / 'model.pt')

# --- Optional CV: location-level bootstrap (set > 0 before final fit) ---
BOOTSTRAP_ROUNDS = 0
if BOOTSTRAP_ROUNDS > 0:
    train_mod.train(
        csv_path=TRAIN_CSV,
        output_path=OUT_PT,
        val_fraction=0.2,
        seed=42,
        bootstrap_rounds=BOOTSTRAP_ROUNDS,
        use_all_data=False,
        dinov2_weights=None,
        temp_steps=800,
    )

# --- Final artifact: gallery = every row in metadata.csv (what you submit) ---
# T is Haversine-tuned on a location holdout; holdout rows stay in the gallery.
train_mod.train(
    csv_path=TRAIN_CSV,
    output_path=OUT_PT,
    val_fraction=0.2,
    seed=42,
    bootstrap_rounds=0,
    use_all_data=True,
    dinov2_weights=None,
    temp_steps=800,
)

p = Path(OUT_PT)
size_mb = p.stat().st_size / 1e6
md5 = hashlib.md5(p.read_bytes()).hexdigest()[:12]
print(f'model.pt  {size_mb:.1f} MB  md5={md5}')
print('Place downloaded file at:', OUT_PT)

if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(OUT_PT)
    print('Browser download started for model.pt')

## 3. Staff evaluator (optional)

Same command as `SUBMISSION.md`. Requires `model.pt` from section 2.


In [ ]:
import subprocess

ev = PROJECT_DIR / 'eval_project_a.py'
r = subprocess.run(
    [
        sys.executable,
        str(ev),
        '--model', str(PROJECT_DIR / 'model.py'),
        '--preprocess', str(PROJECT_DIR / 'preprocess.py'),
        '--weights', str(PROJECT_DIR / 'model.pt'),
        '--csv', str(REF_CSV),
    ],
    cwd=str(REPO_ROOT),
)
print('eval exit code:', r.returncode)


## 4. Optional: reference set map (current `model.pt`)


In [ ]:
import matplotlib.pyplot as plt
from model import Model  # noqa: E402
from preprocess import prepare_data  # noqa: E402

m = Model(weights_path=str(PROJECT_DIR / 'model.pt')).to(device)
m.eval()
X_ref, y_ref_t = prepare_data(REF_CSV)
with torch.no_grad():
    pred = m.predict(list(X_ref)).numpy()
act = y_ref_t.numpy()
plt.figure(figsize=(6, 5))
plt.scatter(act[:, 1], act[:, 0], c='blue', s=80, label='actual')
plt.scatter(pred[:, 1], pred[:, 0], c='red', s=80, marker='x', label='pred')
for i in range(len(act)):
    plt.plot([act[i, 1], pred[i, 1]], [act[i, 0], pred[i, 0]], 'k-', lw=0.5, alpha=0.5)
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend()
plt.title('Reference set: actual vs predicted')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
